# Audio Probe + Steering Sandbox

This notebook does three things:
1. Pulls a small sample of general sounds from local AudioSet and noise-only clips from local speech-noise.
2. Extracts audio embeddings from a pretrained audio model and trains a linear probe (`sound` vs `noise`).
3. Uses the probe direction as a steering vector in embedding space to inspect how model scores shift.

In [ ]:
# If needed, uncomment:
# %pip install -q datasets transformers scikit-learn librosa soundfile matplotlib

In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import librosa
import matplotlib.pyplot as plt

from datasets import load_dataset
from transformers import AutoProcessor, ClapModel
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

BASE_DIR = Path('.').resolve()
AUDIOSET_PATH = BASE_DIR / 'datasets' / 'AudioSet'
SPEECH_NOISE_PATH = BASE_DIR / 'datasets' / 'speech-noise-dataset'

N_SOUND = 64
N_NOISE = 64
CLIP_SECONDS = 5

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
print('AudioSet path exists:', AUDIOSET_PATH.exists())
print('speech-noise path exists:', SPEECH_NOISE_PATH.exists())

In [ ]:
audioset = load_dataset(str(AUDIOSET_PATH), 'balanced', split='train')
speech_noise = load_dataset(str(SPEECH_NOISE_PATH), split='train')

noise_ds = speech_noise.filter(lambda x: x['label'] == 'noise_only')

sound_idx = np.random.choice(len(audioset), size=min(N_SOUND, len(audioset)), replace=False)
noise_idx = np.random.choice(len(noise_ds), size=min(N_NOISE, len(noise_ds)), replace=False)

sound_rows = audioset.select(sound_idx.tolist())
noise_rows = noise_ds.select(noise_idx.tolist())

print(f'Sound clips sampled: {len(sound_rows)}')
print(f'Noise clips sampled: {len(noise_rows)}')

In [ ]:
def to_fixed_clip(audio_obj, target_sr=48000, clip_seconds=5):
    arr = np.asarray(audio_obj['array'], dtype=np.float32)
    sr = int(audio_obj['sampling_rate'])

    if sr != target_sr:
        arr = librosa.resample(arr, orig_sr=sr, target_sr=target_sr)

    needed = target_sr * clip_seconds
    if len(arr) < needed:
        pad = np.zeros(needed - len(arr), dtype=np.float32)
        arr = np.concatenate([arr, pad])
    else:
        start = 0 if len(arr) == needed else np.random.randint(0, len(arr) - needed + 1)
        arr = arr[start:start + needed]

    return arr.astype(np.float32), target_sr

X_audio = []
y = []
meta = []

for ex in sound_rows:
    clip, sr = to_fixed_clip(ex['audio'], target_sr=48000, clip_seconds=CLIP_SECONDS)
    X_audio.append(clip)
    y.append(1)
    meta.append({'source': 'audioset', 'labels': ex.get('human_labels', [])})

for ex in noise_rows:
    clip, sr = to_fixed_clip(ex['audio'], target_sr=48000, clip_seconds=CLIP_SECONDS)
    X_audio.append(clip)
    y.append(0)
    meta.append({'source': 'speech-noise', 'labels': [ex.get('label', 'noise_only')]})

X_audio = np.stack(X_audio)
y = np.asarray(y, dtype=np.int64)
meta_df = pd.DataFrame(meta)

print('Audio tensor shape:', X_audio.shape)
print('Label balance:', {int(k): int(v) for k, v in zip(*np.unique(y, return_counts=True))})
meta_df.head()

In [ ]:
MODEL_ID = 'laion/clap-htsat-unfused'
processor = AutoProcessor.from_pretrained(MODEL_ID)
clap = ClapModel.from_pretrained(MODEL_ID).to(device)
clap.eval()

@torch.no_grad()
def get_audio_embeddings(audio_batch_np, sr=48000, batch_size=8):
    embs = []
    for i in range(0, len(audio_batch_np), batch_size):
        chunk = [a for a in audio_batch_np[i:i + batch_size]]
        inputs = processor(audios=chunk, sampling_rate=sr, return_tensors='pt', padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        feat = clap.get_audio_features(**inputs)
        feat = torch.nn.functional.normalize(feat, dim=-1)
        embs.append(feat.detach().cpu().numpy())
    return np.concatenate(embs, axis=0)

X_emb = get_audio_embeddings(X_audio, sr=48000, batch_size=8)
print('Embedding shape:', X_emb.shape)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_emb, y, test_size=0.25, random_state=SEED, stratify=y
)

probe = LogisticRegression(max_iter=3000, random_state=SEED)
probe.fit(X_train, y_train)

test_probs = probe.predict_proba(X_test)[:, 1]
test_pred = (test_probs >= 0.5).astype(int)

acc = accuracy_score(y_test, test_pred)
auc = roc_auc_score(y_test, test_probs)
print(f'Probe test accuracy: {acc:.3f}')
print(f'Probe test AUROC:    {auc:.3f}')
print(classification_report(y_test, test_pred, target_names=['noise', 'sound']))

w = probe.coef_[0].astype(np.float32)
w = w / (np.linalg.norm(w) + 1e-8)
b = float(probe.intercept_[0])
print('Steering vector dimension:', w.shape)

In [ ]:
@torch.no_grad()
def get_text_embeddings(texts):
    tok = processor(text=texts, return_tensors='pt', padding=True)
    tok = {k: v.to(device) for k, v in tok.items()}
    t = clap.get_text_features(**tok)
    t = torch.nn.functional.normalize(t, dim=-1)
    return t.detach().cpu().numpy()

texts = ['environmental noise', 'natural sound', 'speech']
text_emb = get_text_embeddings(texts)

def score_against_texts(audio_emb):
    ae = audio_emb / (np.linalg.norm(audio_emb, axis=-1, keepdims=True) + 1e-8)
    return ae @ text_emb.T

alphas = np.array([-3.0, -2.0, -1.0, 0.0, 1.0, 2.0, 3.0], dtype=np.float32)
base_scores = probe.decision_function(X_test)

rows = []
for a in alphas:
    X_shift = X_test + a * w[None, :]
    probe_shift = probe.decision_function(X_shift)
    sim_shift = score_against_texts(X_shift)
    rows.append({
        'alpha': float(a),
        'mean_probe_score': float(np.mean(probe_shift)),
        'delta_probe_score': float(np.mean(probe_shift - base_scores)),
        'sim_noise': float(np.mean(sim_shift[:, 0])),
        'sim_sound': float(np.mean(sim_shift[:, 1])),
        'sim_speech': float(np.mean(sim_shift[:, 2])),
    })

steer_df = pd.DataFrame(rows)
display(steer_df)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(steer_df['alpha'], steer_df['mean_probe_score'], marker='o')
axes[0].set_title('Probe Score vs Steering Strength')
axes[0].set_xlabel('alpha')
axes[0].set_ylabel('mean probe decision score')

axes[1].plot(steer_df['alpha'], steer_df['sim_noise'], marker='o', label='noise text')
axes[1].plot(steer_df['alpha'], steer_df['sim_sound'], marker='o', label='sound text')
axes[1].plot(steer_df['alpha'], steer_df['sim_speech'], marker='o', label='speech text')
axes[1].set_title('CLAP Text Similarity Shift')
axes[1].set_xlabel('alpha')
axes[1].set_ylabel('mean cosine similarity')
axes[1].legend()

plt.tight_layout()
plt.show()

## Notes
- Positive `alpha` moves embeddings toward the learned `sound` probe direction.
- Negative `alpha` moves embeddings toward the opposite (noise-like) direction.
- If the probe is meaningful, you should see monotonic shifts in probe score and text similarities.

To extend this:
- Train probes per AudioSet super-class (music, speech, machinery) instead of one binary probe.
- Re-run with different base models (e.g. larger CLAP variants) and compare steering sensitivity.